In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Locate the repo root without importing from src yet.
_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (_current, *_current.parents)
    if (candidate / "AGENTS.md").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split, baseline_mean_metrics


/Users/yangjaehoon/Desktop/StockLens/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
X_train, y_train = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")
X_val, y_val = load_split("validation", processed_dir=PROJECT_ROOT / "data" / "processed")

feature_cols = list(X_train.columns)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)


Train: (1895, 25)
Validation: (600, 25)


## 1. Refit the same XGBoost model, explain it with SHAP

SHAP values are computed on **train only** -- explaining what the
model learned from train, same as the impurity importance step.

In [4]:
XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
)

xgb_model = XGBRegressor(**XGB_PARAMS)
xgb_model.fit(X_train, y_train)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance_df = (
    pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs_shap,
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance_df


,feature,mean_abs_shap
0,sma_60,0.010864
1,volatility_20,0.010037
2,sma_5,0.006918
3,atr_14,0.006774
4,volume_sma_20,0.006254
5,return_20d,0.005211
6,macd_signal,0.005070
7,sma_20,0.004623
8,price_to_sma_60,0.004025
9,price_to_sma_5,0.003330


## 2. Top-K subsets vs. the train-mean baseline

Same evaluation protocol as the other Embedded methods: refit on
train with only the top-K SHAP features, evaluate once on validation.

In [5]:
top_k_list = [5, 10, 15, 20]

results = [baseline_mean_metrics(y_train, y_val)]

for k in top_k_list:
    top_features = shap_importance_df["feature"].head(k).tolist()

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_train[top_features], y_train)

    y_pred = model.predict(X_val[top_features])

    mse = mean_squared_error(y_val, y_pred)

    results.append({
        "method": "SHAP (XGBoost)",
        "n_selected_features": k,
        "selected_features": top_features,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred),
    })

shap_results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
shap_results_df


,method,n_selected_features,selected_features,RMSE,MAE,R2
0,Baseline (predict train mean),0,[],0.105132,0.077442,-0.008612
1,SHAP (XGBoost),20,"[sma_60, volatility_20, sma_5, atr_14, volume_...",0.109305,0.082123,-0.090265
2,SHAP (XGBoost),5,"[sma_60, volatility_20, sma_5, atr_14, volume_...",0.110841,0.083476,-0.121119
3,SHAP (XGBoost),15,"[sma_60, volatility_20, sma_5, atr_14, volume_...",0.111614,0.084337,-0.136813
4,SHAP (XGBoost),10,"[sma_60, volatility_20, sma_5, atr_14, volume_...",0.112767,0.085732,-0.160417


## 3. Does SHAP agree with gain-based importance, and with RF/LightGBM?

In [6]:
results_dir = PROJECT_ROOT / "data" / "processed" / "embedded_results"

shap_top10 = set(shap_importance_df["feature"].head(10))

paths = {
    "XGBoost (gain)": results_dir / "xgboost_importance_full.csv",
    "RandomForest": results_dir / "random_forest_importance_full.csv",
    "LightGBM": results_dir / "lightgbm_importance_full.csv",
}

other_top10 = {}
for name, path in paths.items():
    if path.exists():
        other_top10[name] = set(pd.read_csv(path)["feature"].head(10))
        overlap = shap_top10 & other_top10[name]
        print(f"SHAP ∩ {name} (top10): {len(overlap)}/10 -> {overlap}")

if len(other_top10) == 3:
    consensus = shap_top10
    for s in other_top10.values():
        consensus &= s
    print("\nAll 4 methods agree (top10):", consensus)


SHAP ∩ XGBoost (gain) (top10): 6/10 -> {'atr_14', 'sma_20', 'return_20d', 'sma_60', 'sma_5', 'volatility_20'}
SHAP ∩ RandomForest (top10): 8/10 -> {'atr_14', 'sma_20', 'price_to_sma_60', 'volume_sma_20', 'sma_60', 'sma_5', 'macd_signal', 'volatility_20'}
SHAP ∩ LightGBM (top10): 7/10 -> {'atr_14', 'price_to_sma_60', 'volume_sma_20', 'sma_60', 'sma_5', 'macd_signal', 'volatility_20'}

All 4 methods agree (top10): {'atr_14', 'volatility_20', 'sma_60', 'sma_5'}


In [7]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "embedded_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

shap_importance_df.to_csv(OUTPUT_DIR / "shap_importance_full.csv", index=False)
shap_results_df.to_csv(OUTPUT_DIR / "shap_importance_results.csv", index=False)

print("Saved to:", OUTPUT_DIR)


Saved to: /Users/yangjaehoon/Desktop/StockLens/data/processed/embedded_results
